# Few-Shot Prompting — Teaching by Example Inside the Prompt

In [ ]:
# If you are running this on Google Colab, uncomment and run the line below first.
# !pip install -q boto3 openai tiktoken anthropic matplotlib

## How LLM calls work in this notebook

Every live API call goes through `LLMRouter` from `garage_helper`. Here is what happens:

1. **Model resolution** — the router maps your model name to the right provider (`anthropic`, `openai`, `azure_openai`, `bedrock_claude`, `gemini`, etc.) automatically.
2. **Provider** — the matching provider class builds the API request body — including any `system` prompt and the `messages` list — and dispatches it. The client is cached and reused across cells.
3. **Few-shot messages** — multi-turn examples are passed as a `messages` list where alternating `user`/`assistant` turns teach the model the pattern. The router forwards them to whichever provider is active.
4. **Return value** — `router.generate()` returns a plain string; `router.generate_response()` returns an `LLMResponse` with `.text`, `.input_tokens`, `.output_tokens`.

These notebooks have been tested with **Claude** (via Anthropic direct API and AWS Bedrock) and **GPT** models (via Azure OpenAI and direct OpenAI). Set `verbose=True` on the router to see the full request trace in cell output.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath("../.."))

from garage_helper import setup_llm, LLMRouter

# Contributors: add DEFAULT_LLM_MODEL (and any provider credentials) to a .env at the repo root.
# Learners: setup_llm() will run an interactive wizard to pick a provider and enter credentials.
MODEL  = setup_llm()
router = LLMRouter(default_model=MODEL, verbose=False)

# Few-shot calls need the provider to accept a raw messages list.
# The helper below wraps router.generate for the common few-shot pattern.
def ask_messages(system: str, messages: list, max_tokens: int = 200) -> str:
    """Flatten few-shot messages into a prompt string understood by the router."""
    # Build a history string from all but the last user turn
    history_parts = []
    for m in messages[:-1]:
        role = m["role"].capitalize()
        history_parts.append(f"{role}: {m['content']}")
    history = "\n".join(history_parts)
    last_user = messages[-1]["content"]
    combined = f"{history}\n\nUser: {last_user}" if history else last_user
    return router.generate(combined, model=MODEL, system=system, max_tokens=max_tokens)

## When telling is not enough — and showing fixes it

In the previous notebook — Zero-Shot Prompting — we saw how specificity, role, constraints, and negative instructions get you surprisingly far without any examples. But every practitioner hits the same wall eventually.

You write a careful zero-shot prompt asking for a particular output format. The model comes back with something close but not quite right — slightly wrong tone, wrong structure, the wrong level of detail. You rewrite the constraint. It shifts in a different direction. You tighten the negative instructions. It ignores one of them. You spend twenty minutes rephrasing the same intent.

Here is what is happening: there are some patterns that are genuinely hard to describe in words but trivially obvious from a single example. The shape of a particular JSON schema, the exact level of terseness in a review comment, the ratio of code to explanation in a tutorial — these things are easy to demonstrate and painful to specify.

That is what few-shot prompting solves. Instead of describing the pattern you want, you show it. Two or three well-chosen input–output pairs and the model snaps into exactly the format you need — no weight updates, no fine-tuning, no training run. It adapts entirely from what is in the context window right now.

## Why showing beats telling — the in-context learning connection

Before we write a single prompt, it is worth understanding *why* this works, because it connects directly back to what the Transformer is doing under the hood.

When the model processes your few-shot examples, the attention mechanism is reading both the inputs and the outputs simultaneously — the full sequence. Each output token attends to the corresponding input token and to all the prior examples. By the time the model reaches your final unlabelled input, the attention weights have been shaped by the pattern demonstrated in the examples. The model's key-value pairs in context act as a temporary, soft lookup table — inputs that look like the examples pull outputs that look like the example responses.

No weights changed. No gradient was computed. The "learning" happened entirely through attention over the context window.

**The analogy:** imagine starting a new job and sitting next to a colleague who has been there three years. You do not attend a training course. Instead, you watch how they handle three customer calls. By the fourth call you answer yourself, you have already inferred the tone, the escalation threshold, the phrasing for common situations — all from watching. That is in-context learning. The training course is fine-tuning. Watching three calls is few-shot.

In [ ]:
zero_shot_messages = [
    {
        "role": "user",
        "content": (
            "Classify the sentiment of this review as a single word — "
            "either POSITIVE, NEGATIVE, or NEUTRAL, then a colon, then a "
            "one-sentence reason. Review: 'The battery dies after four hours.'"
        )
    }
]

few_shot_messages = [
    {"role": "user",      "content": "Review: 'Absolutely love the keyboard feel.'"},
    {"role": "assistant", "content": "POSITIVE: The reviewer highlights a specific physical feature as a strong point."},
    {"role": "user",      "content": "Review: 'Arrived on time. Does what it says.'"},
    {"role": "assistant", "content": "NEUTRAL: No strong praise or complaint — functional satisfaction only."},
    {"role": "user",      "content": "Review: 'Screen flickered twice in the first week.'"},
    {"role": "assistant", "content": "NEGATIVE: A hardware defect appeared early, signalling reliability concerns."},
    {"role": "user",      "content": "Review: 'The battery dies after four hours.'"},
]

system = "You are a precise product review classifier."

print("ZERO-SHOT response:")
print(ask_messages(system, zero_shot_messages))

print()
print("FEW-SHOT response:")
print(ask_messages(system, few_shot_messages))

print()
print("Few-shot snapped the format into place without any description of what the format is.")

## How many examples? — the count matters less than you think

The original few-shot paper (GPT-3, Brown et al. 2020) showed diminishing returns beyond roughly eight examples for most tasks. In practice, the sweet spot for modern models is usually **two to five**. That sounds surprisingly small, but it makes sense when you think about what the examples are doing: they are demonstrating a pattern, not training a model. Once the pattern is unambiguous, more examples add tokens without adding signal.

**The analogy:** you are teaching someone a new card game. You play out two hands face-up and explain what you are doing. By the third hand they have the rules. Playing another ten face-up hands does not make them understand the rules any better — it just delays the actual game. Three examples demonstrate the pattern. Thirty examples crowd the context window.

The practical heuristic: start with three examples. Add a fourth only if the model is still making a specific systematic mistake. Add a fifth if the fourth fixed it but a new edge case appeared. Beyond five, look at whether the problem is actually a few-shot problem — it might be a fine-tuning problem.

In [ ]:
examples_pool = [
    ("We fixed a crash that happened when users opened a file larger than 2GB.",
     "[FIX] file-loader: prevent crash on files > 2GB"),
    ("Added a dark mode toggle to the settings page.",
     "[FEAT] settings: add dark mode toggle"),
    ("Improved search query speed by 40% by adding a composite index.",
     "[PERF] search: add composite index, 40% query speedup"),
    ("Removed the deprecated v1 authentication endpoint.",
     "[BREAKING] auth: remove deprecated v1 endpoint"),
    ("Updated the README with local setup instructions.",
     "[DOCS] readme: add local setup instructions"),
]

target_input = "Fixed a memory leak in the WebSocket connection handler that caused servers to crash after 24 hours."
system = "You are a changelog formatter. Output exactly one line — nothing else."

def build_messages(n_examples: int) -> list:
    messages = []
    for user_text, assistant_text in examples_pool[:n_examples]:
        messages.append({"role": "user",      "content": user_text})
        messages.append({"role": "assistant", "content": assistant_text})
    messages.append({"role": "user", "content": target_input})
    return messages

print(f"Target input: '{target_input}'")
print("=" * 60)

for n in [0, 1, 2, 3, 5]:
    messages = build_messages(n)
    output = ask_messages(system, messages, max_tokens=80).strip()
    approx_tokens = sum(len(m["content"]) for m in messages) // 4
    print(f"  {n}-shot (~{approx_tokens:>4} input tokens):  {output}")

print()
print("Notice: 2-3 shots usually produce correct format. More examples → more tokens, not better output.")

## Example selection — not all examples are equal

Picking random examples from your dataset is rarely the right move. The model's attention mechanism gives more weight to examples that are semantically similar to the target input — so examples that are close to the target in topic, structure, and complexity will activate the pattern more reliably than examples chosen for variety alone.

**The analogy:** you are about to negotiate a salary. You ask three colleagues what they said in their negotiations. If one of them negotiated in a completely different city, different role, and five years ago, their example is weakly relevant. The colleague who negotiated at the same company, same level, last quarter is the one whose script you actually want to hear. Relevance to the specific target beats broad coverage.

Three selection principles that actually help:
1. **Semantic proximity** — examples that are topically similar to the target.
2. **Structural coverage** — if the output can have multiple types or branches, cover the relevant ones.
3. **Edge case representation** — if you know there is a failure mode, include an example that demonstrates the correct handling of it.

In [ ]:
target = "I keep getting a 429 error when calling your API. My code was working yesterday."

irrelevant_examples = [
    ("I was charged twice for my subscription this month.", "BILLING"),
    ("Can I change the email address on my account?",       "ACCOUNT"),
    ("Where can I find your refund policy?",                "GENERAL"),
]

relevant_examples = [
    ("The SDK throws a ConnectionTimeout after 30 seconds. My network is fine.",  "TECHNICAL"),
    ("Authentication fails even though I copied the API key from the dashboard.",  "TECHNICAL"),
    ("I upgraded to v2 of the API and now my POST requests return 422.",           "TECHNICAL"),
]

system = "Classify the support ticket. Reply with exactly one word: BILLING, TECHNICAL, ACCOUNT, or GENERAL."

def run(examples: list, label: str) -> str:
    messages = []
    for user_text, category in examples:
        messages.append({"role": "user",      "content": user_text})
        messages.append({"role": "assistant", "content": category})
    messages.append({"role": "user", "content": target})
    answer = ask_messages(system, messages, max_tokens=10).strip()
    correct = "✓" if answer == "TECHNICAL" else "✗"
    print(f"  {label:<28}  →  {answer:<12}  {correct}")
    return answer

print(f"Target ticket: '{target}'")
print(f"Expected: TECHNICAL")
print()
print(f"  {'Example set':<28}     {'Answer':<12}  Correct?")
print("-" * 58)
run(irrelevant_examples, "3 irrelevant examples")
run(relevant_examples,   "3 relevant examples")

print()
print("Both sets use the same number of examples and the same format.")
print("Relevance to the target is what drives accuracy — not just count.")

## Ordering effects — the last example is the loudest

The order of your examples is not neutral. Because of the recency bias in attention — later tokens attend to recent context with higher weight — the example immediately before your target input has the most influence on the output. This is the same mechanism responsible for the "lost in the middle" effect mentioned in the Anatomy of a Prompt notebook.

**The analogy:** think about how a judge scores an Olympic diving competition. The last diver they watched is the freshest in memory. If that diver was excellent, the judge's benchmark shifts up; if they were sloppy, it shifts down. The earlier divers matter, but the final one anchors the scale. Your examples work the same way — the last one in the list sets the strongest prior for the response.

The practical consequence: put your most representative, cleanest, most on-format example last. Do not end with an edge case or an unusual variant — save that for the middle.

In [ ]:
clean_example  = ("Adds null check before calling user.save()",
                  "Add null check before calling user.save()")
sloppy_example = ("there was a missing validation step on the email field so I added it",
                  "Added missing validation for email field in the sign-up form controller")
target_diff    = "The logout button was missing from the mobile nav menu so I put it back."

system = (
    "You are a commit message formatter. "
    "Output exactly one commit message — nothing else."
)

def run_ordered(examples: list, label: str) -> str:
    messages = []
    for user_text, assistant_text in examples:
        messages.append({"role": "user",      "content": user_text})
        messages.append({"role": "assistant", "content": assistant_text})
    messages.append({"role": "user", "content": target_diff})
    output = ask_messages(system, messages, max_tokens=60).strip()
    tense_ok  = not output.lower().startswith(("added", "fixed", "removed", "updated", "changed"))
    length_ok = len(output) <= 50
    print(f"  [{label}]")
    print(f"    Output        : '{output}'")
    print(f"    Imperative    : {'✓' if tense_ok  else '✗ (past tense)'}")
    print(f"    Under 50 chars: {'✓' if length_ok else '✗ (' + str(len(output)) + ' chars)'}")
    return output

print(f"Target: '{target_diff}'")
print()
run_ordered([clean_example,  sloppy_example], "clean first, sloppy last  ← worst order")
print()
run_ordered([sloppy_example, clean_example],  "sloppy first, clean last  ← best order")

print()
print("Same two examples, same model. Order changes which example anchors the output.")
print("Put your best, most canonical example immediately before the target.")

## Putting it together — a production-ready few-shot template

Combining all three principles: choose examples close to your target, cover the structural variants you care about, and put the cleanest example last. Here is how that looks for a real task — extracting structured data from unstructured support messages.

In [ ]:
import json

system = (
    "You are a support ticket parser. "
    "Output only valid JSON — no commentary, no markdown fences."
)

examples = [
    (
        "Hi, I think I was billed twice in March. I see two charges of $49 on my credit card.",
        json.dumps({"issue_type": "BILLING", "severity": "HIGH",
                    "product": "subscription", "summary": "Duplicate charge in March"})
    ),
    (
        "I can't log into my account. I reset my password but the email never arrived.",
        json.dumps({"issue_type": "ACCOUNT", "severity": "HIGH",
                    "product": "auth", "summary": "Password reset email not delivered"})
    ),
    (
        "The export to CSV button does nothing when I click it. No error, just nothing happens. Chrome on Mac.",
        json.dumps({"issue_type": "TECHNICAL", "severity": "MEDIUM",
                    "product": "dashboard", "summary": "CSV export button unresponsive on Chrome/Mac"})
    ),
]

target = (
    "Whenever I try to generate a report with more than 500 rows it just spins forever "
    "and never finishes. Smaller reports work fine. This is blocking our end-of-month process."
)

messages = []
for user_text, assistant_text in examples:
    messages.append({"role": "user",      "content": user_text})
    messages.append({"role": "assistant", "content": assistant_text})
messages.append({"role": "user", "content": target})

raw_output = ask_messages(system, messages, max_tokens=150).strip()

print(f"Target message:\n  '{target}'")
print()
print("Raw model output:")
print(raw_output)

try:
    parsed = json.loads(raw_output)
    print()
    print("Parsed successfully — fields extracted:")
    for k, v in parsed.items():
        print(f"  {k:<14}: {v}")
except json.JSONDecodeError:
    print("\nJSON parse failed — output not clean JSON.")

## The foundations link — why this works without weight updates

It is worth pausing here because what few-shot prompting does is genuinely surprising the first time you think about it carefully.

In the Transformer Internals notebook we saw that each attention head computes a weighted sum over all tokens in the sequence. When you add few-shot examples to the context, you are adding tokens that the model's attention mechanism reads alongside your target input. The query vectors from the target input attend to the key vectors of the examples, and if the target resembles the example inputs, it pulls the corresponding value vectors — which encode the pattern of the example outputs — strongly into the computation.

This is sometimes called **in-context learning** (ICL). It is not learning in the weight-update sense. The model's parameters are frozen. What is happening is more like pattern completion: the attention mechanism is recognising a template and completing it. The examples you provide shift the probability distribution over next tokens without changing a single number in the model.

That is both the power and the limit of few-shot prompting. The pattern has to fit in the context window. It does not persist between calls. And it works best when the pattern you want is actually present — in some compressed form — in the model's pretraining data. For patterns the model has never seen at all, you need fine-tuning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualise how few-shot examples shift the token probability distribution
# for a simple sentiment classification: POSITIVE vs NEGATIVE vs NEUTRAL

np.random.seed(7)

labels = ["POSITIVE", "NEGATIVE", "NEUTRAL"]

# Simulated log-probability distributions (unnormalised, then softmaxed)
# Target review: "Overpriced for what you get."
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

scenarios = {
    "Zero-shot": softmax(np.array([-0.5, 0.1, 0.4])),     # model hedges toward NEUTRAL
    "1-shot\n(one positive example)": softmax(np.array([0.6, -0.3, 0.1])),  # anchored wrong
    "3-shot\n(balanced relevant)": softmax(np.array([-1.1, 2.2, -0.3])),    # correctly NEGATIVE
}

x = np.arange(len(labels))
width = 0.25
colours = ["steelblue", "#d4a843", "seagreen"]

fig, ax = plt.subplots(figsize=(9, 4))

for i, (scenario, probs) in enumerate(scenarios.items()):
    offset = (i - 1) * width
    bars = ax.bar(x + offset, probs, width, label=scenario,
                  color=colours[i], alpha=0.88, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel("Approximate token probability")
ax.set_title("How few-shot examples shift the output distribution\n"
             "Target: 'Overpriced for what you get.'")
ax.set_ylim(0, 1.0)
ax.legend(fontsize=9)
ax.axhline(0.333, color="grey", linewidth=0.7, linestyle="--", alpha=0.5, label="uniform prior")
plt.tight_layout()
plt.show()

print("Zero-shot: model is uncertain — probability spread across all three labels.")
print("1-shot with irrelevant example: examples anchor the wrong direction.")
print("3-shot with relevant examples: probability mass concentrates on NEGATIVE.")
print("No weights changed between scenarios. The examples did all the shifting.")

## When few-shot breaks down — and what to do instead

Few-shot is not a universal fix. Knowing when it stops helping is as important as knowing how to apply it.

In [ ]:
system = "You are a helpful assistant. Be direct."

cases = [
    {
        "label": "FORMAT task — few-shot wins",
        "description": "Reformat unstructured data into a fixed schema. Pattern is demonstrable.",
        "messages": [
            {"role": "user",      "content": "John Smith, 34, Engineer at Acme"},
            {"role": "assistant", "content": "name: John Smith | age: 34 | role: Engineer | company: Acme"},
            {"role": "user",      "content": "Priya Nair, 28, Designer at Bloom Studio"},
            {"role": "assistant", "content": "name: Priya Nair | age: 28 | role: Designer | company: Bloom Studio"},
            {"role": "user",      "content": "Carlos Ruiz, 41, CTO at Neon Systems"},
        ]
    },
    {
        "label": "REASONING task — few-shot alone is not enough",
        "description": "Multi-step maths. The model needs to show its work, not pattern-match a format.",
        "messages": [
            {"role": "user",      "content": "A train travels 60 km at 30 km/h. How long does it take?"},
            {"role": "assistant", "content": "2 hours"},
            {"role": "user",      "content": "A car covers 150 km in 2.5 hours. What is its average speed?"},
            {"role": "assistant", "content": "60 km/h"},
            {"role": "user",      "content": "A cyclist rides at 20 km/h for 45 minutes, then at 15 km/h for 30 minutes. What is the total distance?"},
        ]
    },
]

for case in cases:
    response = ask_messages(system, case["messages"], max_tokens=120)
    print(f"[{case['label']}]")
    print(f"  Task type  : {case['description']}")
    print(f"  Response   : {response.strip()}")
    print()

print("Few-shot excels at format and classification — structured, demonstrable patterns.")
print("For multi-step reasoning, showing answer-only examples doesn't help the model reason.")
print("That is where Chain-of-Thought prompting takes over — the next section.")

## Putting it all together — the few-shot decision framework

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Visualise where few-shot sits in the prompting landscape vs zero-shot and CoT

categories = [
    "Format\nconversion",
    "Classification",
    "Style\ntransfer",
    "Multi-step\nreasoning",
    "Novel fact\nretrieval",
]

# Effectiveness scores (0–1) for each technique on each task category
zero_shot  = [0.55, 0.65, 0.50, 0.60, 0.70]
few_shot   = [0.95, 0.90, 0.92, 0.65, 0.70]
cot        = [0.60, 0.70, 0.55, 0.95, 0.72]

x = np.arange(len(categories))
width = 0.26

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(x - width, zero_shot, width, label="Zero-shot",  color="steelblue", alpha=0.85)
ax.bar(x,         few_shot,  width, label="Few-shot",   color="seagreen",  alpha=0.85)
ax.bar(x + width, cot,       width, label="Chain-of-Thought (next notebook)",
       color="tomato", alpha=0.85)

ax.set_ylabel("Relative effectiveness (illustrative)")
ax.set_title("Prompting technique effectiveness by task type")
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=10)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=9)
ax.axhline(0.8, color="grey", linewidth=0.6, linestyle="--", alpha=0.4)
ax.text(len(categories) - 0.1, 0.82, "good enough", fontsize=8, color="grey", ha="right")

plt.tight_layout()
plt.show()

print("Few-shot dominates on format, classification, and style tasks.")
print("Chain-of-Thought takes over on multi-step reasoning — that is next.")
print("Neither technique helps when the model simply lacks the factual knowledge.")

## Key takeaways

- **Showing beats telling** for structured patterns — when the output format is hard to describe precisely in words, two or three examples snap the model into it instantly.
- **In-context learning** works through attention: your examples shift the model's output probability distribution without updating a single weight. The Transformer reads the pattern and completes it.
- **Two to five examples** is usually the sweet spot. Beyond five you spend tokens on diminishing returns — if more examples keep failing, the problem is likely a fine-tuning job, not a prompting one.
- **Example selection matters**: semantically close examples outperform random or diverse ones. Pick examples that look like your target input.
- **Order matters**: the example immediately before the target anchors the output most strongly. Put your cleanest, most canonical example last.
- **Few-shot does not fix reasoning gaps** — showing answer-only examples does not help the model work through a multi-step problem. That is what Chain-of-Thought is for.
- **Few-shot does not add knowledge** — if the model's pretraining did not include something, examples in the prompt will not teach it that fact.

---

Next up: **Controlling the Output** — now that you can teach the model a pattern, let's look at how to govern exactly what it produces: format, length, tone, structured JSON, stop sequences, and the temperature dial.